In [1]:
from RNN_Sequences.chicago_transit_data import load_ridership_data
from data_loaders.TimeSeriesDataset import TimeSeriesDataset
from datetime import datetime, timedelta
from matplotlib import pyplot as plt
from models.rnn_models import SimpleRnnModel
from torch.utils.data import DataLoader
import pandas as pd
import torch
import torchmetrics

data_scale = 1e6

df_original = load_ridership_data()
df = df_original[["rail"]] / data_scale
df["next_day_type"] = df_original["day_type"].shift(-1)
df = pd.get_dummies(df, dtype=float)
print("Min: ", df.index.min())
print("Max: ", df.index.max())
df


Min:  2001-01-01 00:00:00
Max:  2026-02-28 00:00:00


,rail,next_day_type_A,next_day_type_U,next_day_type_W
date,,,,
2001-01-01,0.126455,0.0,0.0,1.0
2001-01-02,0.501952,0.0,0.0,1.0
2001-01-03,0.536432,0.0,0.0,1.0
2001-01-04,0.550011,0.0,0.0,1.0
2001-01-05,0.557917,1.0,0.0,0.0
...,...,...,...,...
2026-02-24,0.416203,0.0,0.0,1.0
2026-02-25,0.425585,0.0,0.0,1.0
2026-02-26,0.421868,0.0,0.0,1.0


In [2]:
def split_dataframe(df, percentage):
    return df.index[int(len(df) * percentage)]

covid_start = datetime(2020, 3, 1)
pre_covid_rows = df[df.index < covid_start]
post_covid_rows = df[df.index > covid_start]

start_date_pre_covid = df.index.min()
start_date_post_covid = covid_start

valid_start_date_pre_covid = split_dataframe(pre_covid_rows, 0.6)
valid_start_date_post_covid = split_dataframe(post_covid_rows, 0.6)

test_start_date_pre_covid = split_dataframe(pre_covid_rows, 0.8)
test_start_date_post_covid = split_dataframe(post_covid_rows, 0.8)

print("Pre-COVID: ", start_date_pre_covid, valid_start_date_pre_covid, test_start_date_pre_covid)
print("Post-COVID: ", start_date_post_covid, valid_start_date_post_covid, test_start_date_post_covid)

Pre-COVID:  2001-01-01 00:00:00 2012-07-01 00:00:00 2016-05-01 00:00:00
Post-COVID:  2020-03-01 00:00:00 2023-10-07 00:00:00 2024-12-18 00:00:00


In [3]:
def slice_df(df, pre_covid_start, pre_covid_end, post_covid_start, post_covid_end):
    return pd.concat([df[pre_covid_start:pre_covid_end], df[post_covid_start: post_covid_end]])

df_train = slice_df(df, start_date_pre_covid, valid_start_date_pre_covid - timedelta(days=1),
                    covid_start, valid_start_date_post_covid - timedelta(days=1))

print(df_train.index.min(), df_train.index.max(), len(df_train))

df_valid = slice_df(df, valid_start_date_pre_covid, test_start_date_pre_covid - timedelta(days=-1),
                    valid_start_date_post_covid, test_start_date_post_covid - timedelta(days=-1))

df_test = slice_df(df, test_start_date_pre_covid, covid_start + timedelta(days=-1),
                   test_start_date_post_covid, df.index.max())


2001-01-01 00:00:00 2023-10-06 00:00:00 5514


In [4]:
window_length = 56

rail_train = torch.FloatTensor(df_train.values)
rail_valid = torch.FloatTensor(df_valid.values)
rail_test = torch.FloatTensor(df_test.values)

target_slice = slice(0, 1)
train_set = TimeSeriesDataset(rail_train, window_length, target_data_slice=target_slice)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
valid_set = TimeSeriesDataset(rail_valid, window_length, target_data_slice=target_slice)
valid_loader = DataLoader(valid_set, batch_size=32)
test_set = TimeSeriesDataset(rail_test, window_length, target_data_slice=target_slice)
test_loader = DataLoader(test_set, batch_size=32)

In [5]:
from model_runner.ModelRunner import ModelRunner
from torch import nn

torch.manual_seed(42)
model = SimpleRnnModel(input_size=4, hidden_size=64, output_size=1)
loss_fn = nn.HuberLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.90)
metric = torchmetrics.MeanSquaredLogError()

runner = ModelRunner(model, metric, optimizer, loss_fn)
runner.train_model(train_loader, valid_loader, n_epochs=50, patience=75)

test_result = runner.test_model(test_loader) * 1e6
print(test_result)

Epoch:1 / 50, train loss: 0.0055, train metric: 0.0059, valid metric: 0.0022 (best),  in 3.5s
Epoch:2 / 50, train loss: 0.0022, train metric: 0.0027, valid metric: 0.0020,  in 3.9s
Epoch:3 / 50, train loss: 0.0020, train metric: 0.0023, valid metric: 0.0020,  in 3.6s
Epoch:4 / 50, train loss: 0.0018, train metric: 0.0021, valid metric: 0.0019,  in 3.9s
Epoch:5 / 50, train loss: 0.0015, train metric: 0.0017, valid metric: 0.0017,  in 4.1s
Epoch:6 / 50, train loss: 0.0011, train metric: 0.0012, valid metric: 0.0018,  in 3.3s
Epoch:7 / 50, train loss: 0.0009, train metric: 0.0009, valid metric: 0.0012,  in 3.3s
Epoch:8 / 50, train loss: 0.0008, train metric: 0.0008, valid metric: 0.0013,  in 3.3s
Epoch:9 / 50, train loss: 0.0008, train metric: 0.0008, valid metric: 0.0013,  in 3.3s
Epoch:10 / 50, train loss: 0.0008, train metric: 0.0008, valid metric: 0.0012,  in 3.4s
Epoch:11 / 50, train loss: 0.0008, train metric: 0.0007, valid metric: 0.0013,  in 3.4s
Epoch:12 / 50, train loss: 0.0007,

In [6]:
# Note that in order to make predictions, we must have the past window_length days of data.
df_test_post_covid = df_test[covid_start:]
test_post_covid_set = TimeSeriesDataset(df_test_post_covid, window_length, target_data_slice=target_slice)
test_post_covid_loader = DataLoader(test_post_covid_set, batch_size=32)

test_predictions = runner.run_model(test_post_covid_loader).cpu()

# df_test_predictions = pd.DataFrame(test_predictions)
# df_test_predictions = df_test_predictions * data_scale
# df_test_predictions.columns = ["predicted_rail"]
#
# predictions_index = df_test_post_covid.index[0:len(df_test_predictions)]
# df_test_predictions.index = predictions_index
# df_test_predictions

InvalidIndexError: (56, slice(0, 1, None))

In [ ]:
df_combined = df_test.join(df_test_predictions, how = "left")

# Plot rail vs predicted_rail
ax1 = df_combined[["rail", "predicted_rail"]].plot(grid=True, marker=".", figsize=(10, 5), title="Rail vs Predicted Rail")
plt.ticklabel_format(style='plain', axis='y')
plt.show()


In [ ]:
# Plot delta (error) for rail
(df_combined["rail"] - df_combined["predicted_rail"]).plot(grid=True, marker=".", figsize=(10, 5), title="Rail Error (Actual - Predicted)")
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.ticklabel_format(style='plain', axis='y')
plt.show()

